# 🕵️ Dia 55 — Lendo logs de autenticação Windows e Linux

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 55* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-55.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

A caixa jura que não abriu o webmail depois do almoço. O log diz que alguém abriu, de 203.0.113.91, e criou uma regra. Rafa: *"Força bruta, spray, ou alguém que já tinha a senha?"*

Neste notebook você vai:

1. **Perfilar o *spray* do `bastion01`** (os cinco comandos do passo 1) com `re` e `Counter`.
2. **Desenhar os mapas de calor do WS-FIN-07 em texto**, sem pandas.
3. **Abrir a janela 02h–04h** e escrever a questão em aberto.
4. **Classificar as origens**: *spray*, força bruta ou poucas falhas numa conta.
5. **Repetir com a semana da `cx.centro07`** (notebook e webmail).
6. **Jogar o quiz dos tipos de *logon*.**

🐍 Só biblioteca padrão. O `heatmap.py` original (pandas + matplotlib) está numa célula opcional.


In [ ]:
# ------------------------------------------------------------
# RECEBER BASTION01-AUTH.LOG (EVID-001) E WS-FIN-07-SECURITY.CSV (EVID-002)
# Plano A: baixar do repositório original (GitHub raw), com urllib.
# Plano B: usar a cópia embutida logo abaixo (o arquivo inteiro,
#          copiado do upstream/), para o notebook rodar sem internet.
# ------------------------------------------------------------
import urllib.request      # baixa arquivos da web, sem instalar nada
import hashlib             # SHA-256: a impressão digital dos bytes

FORCAR_OFFLINE = False     # True = nem tenta a rede (útil em sala sem Wi-Fi)
URL_BASE = 'https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-4-digital-forensics-incident-investigation/resources/'

RESERVA = {}             # caminho do arquivo -> conteúdo embutido (plano B)
RESERVA['case-lab-p4/logs/bastion01-auth.log'] = ''.join((
    'Mar 13 14:02:11 bastion01 sshd[21807]: Accepted publickey for ops.admin from 10.10.50.12 port 51544 ssh2: ED25519 SHA256:q8m1Xx0lab0synthetic0key0fingerprint0AAAA\n'
    'Mar 13 14:02:12 bastion01 sshd[21807]: pam_unix(sshd:session): session opened for user ops.admin(uid=1001) by (uid=0)\n'
    'Mar 13 16:47:39 bastion01 sshd[21814]: Accepted publickey for ops.admin from 10.10.50.12 port 52010 ssh2: ED25519 SHA256:q8m1Xx0lab0synthetic0key0fingerprint0AAAA\n'
    'Mar 13 16:47:40 bastion01 sshd[21814]: pam_unix(sshd:session): session opened for user ops.admin(uid=1001) by (uid=0)\n'
    'Mar 14 02:51:03 bastion01 sshd[21815]: Failed password for invalid user admin from 203.0.113.45 port 49208 ssh2\n'
    'Mar 14 02:51:15 bastion01 sshd[21816]: Failed password for root from 203.0.113.45 port 49212 ssh2\n'
    'Mar 14 02:51:24 bastion01 sshd[21817]: Failed password for invalid user backup from 203.0.113.45 port 49216 ssh2\n'
    'Mar 14 02:51:34 bastion01 sshd[21818]: Failed password for invalid user oracle from 203.0.113.45 port 49224 ssh2\n'
    'Mar 14 02:51:49 bastion01 sshd[21819]: Failed password for svc_backup from 203.0.113.45 port 49230 ssh2\n'
    'Mar 14 02:52:04 bastion01 sshd[21820]: Failed password for invalid user test from 203.0.113.45 port 49237 ssh2\n'
    'Mar 14 02:52:18 bastion01 sshd[21821]: Failed password for invalid user ubuntu from 203.0.113.45 port 49245 ssh2\n'
    'Mar 14 02:52:30 bastion01 sshd[21822]: Failed password for invalid user ftpuser from 203.0.113.45 port 49249 ssh2\n'
    'Mar 14 02:52:39 bastion01 sshd[21823]: Failed password for invalid user admin from 203.0.113.45 port 49252 ssh2\n'
    'Mar 14 02:52:52 bastion01 sshd[21824]: Failed password for root from 203.0.113.45 port 49256 ssh2\n'
    'Mar 14 02:53:02 bastion01 sshd[21825]: Failed password for invalid user backup from 203.0.113.45 port 49264 ssh2\n'
    'Mar 14 02:53:16 bastion01 sshd[21826]: Failed password for invalid user oracle from 203.0.113.45 port 49268 ssh2\n'
    'Mar 14 02:53:29 bastion01 sshd[21827]: Failed password for svc_backup from 203.0.113.45 port 49269 ssh2\n'
    'Mar 14 02:53:42 bastion01 sshd[21828]: Failed password for invalid user test from 203.0.113.45 port 49278 ssh2\n'
    'Mar 14 02:53:51 bastion01 sshd[21829]: Failed password for invalid user ubuntu from 203.0.113.45 port 49279 ssh2\n'
    'Mar 14 02:54:03 bastion01 sshd[21830]: Failed password for invalid user ftpuser from 203.0.113.45 port 49287 ssh2\n'
    'Mar 14 02:54:18 bastion01 sshd[21831]: Failed password for invalid user admin from 203.0.113.45 port 49296 ssh2\n'
    'Mar 14 02:54:32 bastion01 sshd[21832]: Failed password for root from 203.0.113.45 port 49303 ssh2\n'
    'Mar 14 02:54:43 bastion01 sshd[21833]: Failed password for invalid user backup from 203.0.113.45 port 49311 ssh2\n'
    'Mar 14 02:54:53 bastion01 sshd[21834]: Failed password for invalid user oracle from 203.0.113.45 port 49316 ssh2\n'
    'Mar 14 02:55:04 bastion01 sshd[21835]: Failed password for svc_backup from 203.0.113.45 port 49321 ssh2\n'
    'Mar 14 02:55:16 bastion01 sshd[21836]: Failed password for invalid user test from 203.0.113.45 port 49330 ssh2\n'
    'Mar 14 02:55:25 bastion01 sshd[21837]: Failed password for invalid user ubuntu from 203.0.113.45 port 49337 ssh2\n'
    'Mar 14 02:55:37 bastion01 sshd[21838]: Failed password for invalid user ftpuser from 203.0.113.45 port 49343 ssh2\n'
    'Mar 14 02:55:50 bastion01 sshd[21839]: Failed password for invalid user admin from 203.0.113.45 port 49347 ssh2\n'
    'Mar 14 02:56:02 bastion01 sshd[21840]: Failed password for root from 203.0.113.45 port 49348 ssh2\n'
    'Mar 14 02:56:14 bastion01 sshd[21841]: Failed password for invalid user backup from 203.0.113.45 port 49357 ssh2\n'
    'Mar 14 02:56:27 bastion01 sshd[21842]: Failed password for invalid user oracle from 203.0.113.45 port 49359 ssh2\n'
    'Mar 14 02:56:42 bastion01 sshd[21843]: Failed password for svc_backup from 203.0.113.45 port 49368 ssh2\n'
    'Mar 14 02:56:54 bastion01 sshd[21844]: Failed password for invalid user test from 203.0.113.45 port 49374 ssh2\n'
    'Mar 14 02:57:08 bastion01 sshd[21845]: Failed password for invalid user ubuntu from 203.0.113.45 port 49376 ssh2\n'
    'Mar 14 02:57:22 bastion01 sshd[21846]: Failed password for invalid user ftpuser from 203.0.113.45 port 49379 ssh2\n'
    'Mar 14 02:57:34 bastion01 sshd[21847]: Failed password for invalid user admin from 203.0.113.45 port 49383 ssh2\n'
    'Mar 14 02:57:45 bastion01 sshd[21848]: Failed password for root from 203.0.113.45 port 49387 ssh2\n'
    'Mar 14 02:57:59 bastion01 sshd[21849]: Failed password for invalid user backup from 203.0.113.45 port 49395 ssh2\n'
    'Mar 14 02:58:11 bastion01 sshd[21850]: Failed password for invalid user oracle from 203.0.113.45 port 49398 ssh2\n'
    'Mar 14 02:58:26 bastion01 sshd[21851]: Failed password for svc_backup from 203.0.113.45 port 49403 ssh2\n'
    'Mar 14 02:58:38 bastion01 sshd[21852]: Failed password for invalid user test from 203.0.113.45 port 49410 ssh2\n'
    'Mar 14 02:58:49 bastion01 sshd[21853]: Failed password for invalid user ubuntu from 203.0.113.45 port 49411 ssh2\n'
    'Mar 14 02:59:03 bastion01 sshd[21854]: Failed password for invalid user ftpuser from 203.0.113.45 port 49419 ssh2\n'
    'Mar 14 02:59:14 bastion01 sshd[21855]: Failed password for invalid user admin from 203.0.113.45 port 49422 ssh2\n'
    'Mar 14 02:59:28 bastion01 sshd[21856]: Failed password for root from 203.0.113.45 port 49430 ssh2\n'
    'Mar 14 02:59:43 bastion01 sshd[21857]: Failed password for invalid user backup from 203.0.113.45 port 49434 ssh2\n'
    'Mar 14 02:59:54 bastion01 sshd[21858]: Failed password for invalid user oracle from 203.0.113.45 port 49436 ssh2\n'
    'Mar 14 03:00:09 bastion01 sshd[21859]: Failed password for svc_backup from 203.0.113.45 port 49440 ssh2\n'
    'Mar 14 03:00:24 bastion01 sshd[21860]: Failed password for invalid user test from 203.0.113.45 port 49443 ssh2\n'
    'Mar 14 03:00:39 bastion01 sshd[21861]: Failed password for invalid user ubuntu from 203.0.113.45 port 49447 ssh2\n'
    'Mar 14 03:00:48 bastion01 sshd[21862]: Failed password for invalid user ftpuser from 203.0.113.45 port 49450 ssh2\n'
    'Mar 14 03:01:02 bastion01 sshd[21863]: Failed password for invalid user admin from 203.0.113.45 port 49458 ssh2\n'
    'Mar 14 03:01:17 bastion01 sshd[21864]: Failed password for root from 203.0.113.45 port 49461 ssh2\n'
    'Mar 14 03:01:29 bastion01 sshd[21865]: Failed password for invalid user backup from 203.0.113.45 port 49470 ssh2\n'
    'Mar 14 03:01:39 bastion01 sshd[21866]: Failed password for invalid user oracle from 203.0.113.45 port 49478 ssh2\n'
    'Mar 14 03:01:53 bastion01 sshd[21867]: Failed password for svc_backup from 203.0.113.45 port 49486 ssh2\n'
    'Mar 14 03:02:08 bastion01 sshd[21868]: Failed password for invalid user test from 203.0.113.45 port 49491 ssh2\n'
    'Mar 14 03:02:23 bastion01 sshd[21869]: Failed password for invalid user ubuntu from 203.0.113.45 port 49499 ssh2\n'
    'Mar 14 03:02:36 bastion01 sshd[21870]: Failed password for invalid user ftpuser from 203.0.113.45 port 49504 ssh2\n'
    'Mar 14 03:02:51 bastion01 sshd[21871]: Failed password for invalid user admin from 203.0.113.45 port 49511 ssh2\n'
    'Mar 14 03:03:03 bastion01 sshd[21872]: Failed password for root from 203.0.113.45 port 49512 ssh2\n'
    'Mar 14 03:03:13 bastion01 sshd[21873]: Failed password for invalid user backup from 203.0.113.45 port 49517 ssh2\n'
    'Mar 14 03:03:28 bastion01 sshd[21874]: Failed password for invalid user oracle from 203.0.113.45 port 49524 ssh2\n'
    'Mar 14 03:03:40 bastion01 sshd[21875]: Failed password for svc_backup from 203.0.113.45 port 49525 ssh2\n'
    'Mar 14 03:03:55 bastion01 sshd[21876]: Failed password for invalid user test from 203.0.113.45 port 49534 ssh2\n'
    'Mar 14 03:04:05 bastion01 sshd[21877]: Failed password for invalid user ubuntu from 203.0.113.45 port 49539 ssh2\n'
    'Mar 14 03:04:17 bastion01 sshd[21878]: Failed password for invalid user ftpuser from 203.0.113.45 port 49547 ssh2\n'
    'Mar 14 03:04:31 bastion01 sshd[21879]: Failed password for invalid user admin from 203.0.113.45 port 49556 ssh2\n'
    'Mar 14 03:04:46 bastion01 sshd[21880]: Failed password for root from 203.0.113.45 port 49561 ssh2\n'
    'Mar 14 03:05:00 bastion01 sshd[21881]: Failed password for invalid user backup from 203.0.113.45 port 49565 ssh2\n'
    'Mar 14 03:05:14 bastion01 sshd[21882]: Failed password for invalid user oracle from 203.0.113.45 port 49568 ssh2\n'
    'Mar 14 03:05:23 bastion01 sshd[21883]: Failed password for svc_backup from 203.0.113.45 port 49576 ssh2\n'
    'Mar 14 03:05:32 bastion01 sshd[21884]: Failed password for invalid user test from 203.0.113.45 port 49577 ssh2\n'
    'Mar 14 03:05:44 bastion01 sshd[21885]: Failed password for invalid user ubuntu from 203.0.113.45 port 49582 ssh2\n'
    'Mar 14 03:05:55 bastion01 sshd[21886]: Failed password for invalid user ftpuser from 203.0.113.45 port 49589 ssh2\n'
    'Mar 14 03:06:06 bastion01 sshd[21887]: Failed password for invalid user admin from 203.0.113.45 port 49593 ssh2\n'
    'Mar 14 03:06:21 bastion01 sshd[21888]: Failed password for root from 203.0.113.45 port 49598 ssh2\n'
    'Mar 14 03:06:34 bastion01 sshd[21889]: Failed password for invalid user backup from 203.0.113.45 port 49602 ssh2\n'
    'Mar 14 03:06:48 bastion01 sshd[21890]: Failed password for invalid user oracle from 203.0.113.45 port 49605 ssh2\n'
    'Mar 14 03:06:57 bastion01 sshd[21891]: Failed password for svc_backup from 203.0.113.45 port 49614 ssh2\n'
    'Mar 14 03:07:07 bastion01 sshd[21892]: Failed password for invalid user test from 203.0.113.45 port 49617 ssh2\n'
    'Mar 14 03:07:18 bastion01 sshd[21893]: Failed password for invalid user ubuntu from 203.0.113.45 port 49625 ssh2\n'
    'Mar 14 03:07:30 bastion01 sshd[21894]: Failed password for invalid user ftpuser from 203.0.113.45 port 49633 ssh2\n'
    'Mar 14 03:07:39 bastion01 sshd[21895]: Failed password for invalid user admin from 203.0.113.45 port 49636 ssh2\n'
    'Mar 14 03:07:51 bastion01 sshd[21896]: Failed password for root from 203.0.113.45 port 49640 ssh2\n'
    'Mar 14 03:08:01 bastion01 sshd[21897]: Failed password for invalid user backup from 203.0.113.45 port 49647 ssh2\n'
    'Mar 14 03:08:10 bastion01 sshd[21898]: Failed password for invalid user oracle from 203.0.113.45 port 49649 ssh2\n'
    'Mar 14 03:08:19 bastion01 sshd[21899]: Failed password for svc_backup from 203.0.113.45 port 49651 ssh2\n'
    'Mar 14 03:08:28 bastion01 sshd[21900]: Failed password for invalid user test from 203.0.113.45 port 49655 ssh2\n'
    'Mar 14 03:08:42 bastion01 sshd[21901]: Failed password for invalid user ubuntu from 203.0.113.45 port 49657 ssh2\n'
    'Mar 14 03:08:54 bastion01 sshd[21902]: Failed password for invalid user ftpuser from 203.0.113.45 port 49658 ssh2\n'
    'Mar 14 03:09:03 bastion01 sshd[21903]: Failed password for invalid user admin from 203.0.113.45 port 49659 ssh2\n'
    'Mar 14 03:09:14 bastion01 sshd[21904]: Failed password for root from 203.0.113.45 port 49664 ssh2\n'
    'Mar 14 03:09:25 bastion01 sshd[21905]: Failed password for invalid user backup from 203.0.113.45 port 49665 ssh2\n'
    'Mar 14 03:09:37 bastion01 sshd[21906]: Failed password for invalid user oracle from 203.0.113.45 port 49673 ssh2\n'
    'Mar 14 03:09:51 bastion01 sshd[21907]: Failed password for svc_backup from 203.0.113.45 port 49681 ssh2\n'
    'Mar 14 03:10:00 bastion01 sshd[21908]: Failed password for invalid user test from 203.0.113.45 port 49689 ssh2\n'
    'Mar 14 03:10:10 bastion01 sshd[21909]: Failed password for invalid user ubuntu from 203.0.113.45 port 49696 ssh2\n'
    'Mar 14 03:10:19 bastion01 sshd[21910]: Failed password for invalid user ftpuser from 203.0.113.45 port 49703 ssh2\n'
    'Mar 14 03:10:33 bastion01 sshd[21911]: Failed password for invalid user admin from 203.0.113.45 port 49712 ssh2\n'
    'Mar 14 03:10:47 bastion01 sshd[21912]: Failed password for root from 203.0.113.45 port 49713 ssh2\n'
    'Mar 14 03:11:02 bastion01 sshd[21913]: Failed password for invalid user backup from 203.0.113.45 port 49720 ssh2\n'
    'Mar 14 03:11:12 bastion01 sshd[21914]: Failed password for invalid user oracle from 203.0.113.45 port 49727 ssh2\n'
    'Mar 14 03:11:22 bastion01 sshd[21915]: Failed password for svc_backup from 203.0.113.45 port 49734 ssh2\n'
    'Mar 14 03:11:36 bastion01 sshd[21916]: Failed password for invalid user test from 203.0.113.45 port 49742 ssh2\n'
    'Mar 14 03:11:49 bastion01 sshd[21917]: Failed password for invalid user ubuntu from 203.0.113.45 port 49744 ssh2\n'
    'Mar 14 03:12:04 bastion01 sshd[21918]: Failed password for invalid user ftpuser from 203.0.113.45 port 49745 ssh2\n'
    'Mar 14 03:12:13 bastion01 sshd[21919]: Failed password for invalid user admin from 203.0.113.45 port 49751 ssh2\n'
    'Mar 14 03:12:23 bastion01 sshd[21920]: Failed password for root from 203.0.113.45 port 49753 ssh2\n'
    'Mar 14 03:12:37 bastion01 sshd[21921]: Failed password for invalid user backup from 203.0.113.45 port 49755 ssh2\n'
    'Mar 14 03:12:47 bastion01 sshd[21922]: Failed password for invalid user oracle from 203.0.113.45 port 49759 ssh2\n'
    'Mar 14 03:13:01 bastion01 sshd[21923]: Failed password for svc_backup from 203.0.113.45 port 49760 ssh2\n'
    'Mar 14 03:13:15 bastion01 sshd[21924]: Failed password for invalid user test from 203.0.113.45 port 49766 ssh2\n'
    'Mar 14 03:13:24 bastion01 sshd[21925]: Failed password for invalid user ubuntu from 203.0.113.45 port 49774 ssh2\n'
    'Mar 14 03:13:36 bastion01 sshd[21926]: Failed password for invalid user ftpuser from 203.0.113.45 port 49783 ssh2\n'
    'Mar 14 03:14:07 bastion01 sshd[21927]: Accepted password for svc_backup from 203.0.113.45 port 50122 ssh2\n'
    'Mar 14 03:14:08 bastion01 sshd[21927]: pam_unix(sshd:session): session opened for user svc_backup(uid=1107) by (uid=0)\n'
    'Mar 14 03:14:08 bastion01 systemd-logind[642]: New session 318 of user svc_backup.\n'
    'Mar 14 03:52:44 bastion01 sshd[21927]: pam_unix(sshd:session): session closed for user svc_backup\n'
))
RESERVA['case-lab-p4/windows/ws-fin-07-security.csv'] = ''.join((
    'TimeCreatedUtc,EventID,TargetUserName,LogonType,IpAddress,WorkstationName\n'
    '2026-03-09T05:24:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-09T10:14:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-09T13:03:23Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-09T13:13:07Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-09T13:51:21Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-09T14:20:23Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-09T14:29:00Z,4625,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-09T15:08:51Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-09T15:47:41Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-09T16:20:19Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-09T16:49:36Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-09T17:04:59Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-09T17:36:39Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-09T17:48:37Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-09T18:34:14Z,4624,acct.lead02,7,-,WS-FIN-07\n'
    '2026-03-09T19:09:22Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-09T19:50:54Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-09T21:20:56Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-09T21:39:22Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-09T21:49:17Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-10T06:42:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-10T14:31:10Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-10T15:00:07Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-10T15:08:10Z,4624,acct.lead02,7,-,WS-FIN-07\n'
    '2026-03-10T15:23:26Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-10T15:44:47Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-10T16:08:00Z,4625,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-10T16:21:17Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-10T16:36:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-10T16:55:35Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-10T17:04:04Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-10T17:25:00Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-10T19:19:06Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-10T19:41:08Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-10T20:06:33Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-10T20:48:43Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-11T12:35:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-11T13:08:15Z,4624,acct.lead02,7,-,WS-FIN-07\n'
    '2026-03-11T13:16:22Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-11T13:17:29Z,4624,acct.lead02,11,-,WS-FIN-07\n'
    '2026-03-11T13:18:42Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T14:25:44Z,4624,acct.lead02,11,-,WS-FIN-07\n'
    '2026-03-11T14:50:25Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-11T15:27:00Z,4625,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-11T15:43:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-11T16:33:55Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T16:39:08Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T16:48:16Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T17:00:12Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-11T17:35:19Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T17:55:17Z,4624,acct.lead02,11,-,WS-FIN-07\n'
    '2026-03-11T18:41:35Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-11T19:08:40Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T19:09:22Z,4624,acct.lead02,7,-,WS-FIN-07\n'
    '2026-03-11T19:10:57Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T19:21:53Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T19:25:43Z,4624,acct.lead02,11,-,WS-FIN-07\n'
    '2026-03-11T20:59:45Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-12T05:04:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-12T06:39:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-12T14:13:21Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-12T14:25:26Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-12T14:55:50Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-12T15:01:16Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-12T15:11:04Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-12T15:27:43Z,4624,acct.lead02,7,-,WS-FIN-07\n'
    '2026-03-12T16:57:07Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-12T18:12:40Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-12T18:28:34Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-12T18:33:25Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-12T18:43:05Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-12T19:39:27Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-12T19:44:51Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-12T19:46:00Z,4625,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-12T20:20:51Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-12T21:01:20Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-13T13:17:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-13T13:27:56Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-13T15:24:20Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-13T16:25:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-13T17:04:28Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-13T17:13:10Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-13T17:45:27Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-13T18:45:06Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-13T19:13:05Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-13T19:28:00Z,4625,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-13T19:31:08Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-13T19:34:17Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-13T20:17:59Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-13T20:32:28Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-13T21:37:18Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-13T21:48:40Z,4624,acct.lead02,11,-,WS-FIN-07\n'
    '2026-03-13T21:50:09Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-13T21:59:56Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-14T02:10:00Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:10:17Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:10:34Z,4625,svc_backup,3,10.10.20.5,-\n'
    '2026-03-14T02:10:51Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:11:08Z,4625,acct.clerk01,3,10.10.20.5,-\n'
    '2026-03-14T02:11:25Z,4625,acct.clerk01,3,10.10.20.5,-\n'
    '2026-03-14T02:11:42Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:11:59Z,4625,acct.clerk01,3,10.10.20.5,-\n'
    '2026-03-14T02:12:16Z,4625,acct.clerk01,3,10.10.20.5,-\n'
    '2026-03-14T02:12:33Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:12:50Z,4625,svc_backup,3,10.10.20.5,-\n'
    '2026-03-14T02:13:07Z,4625,acct.clerk01,3,10.10.20.5,-\n'
    '2026-03-14T02:13:24Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:13:41Z,4625,acct.clerk01,3,10.10.20.5,-\n'
    '2026-03-14T02:13:58Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:14:15Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:14:32Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:14:49Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:15:06Z,4625,svc_backup,3,10.10.20.5,-\n'
    '2026-03-14T02:15:23Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:15:40Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:15:57Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:16:14Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T03:27:41Z,4624,svc_backup,3,10.10.30.17,-\n'
    '2026-03-14T03:27:41Z,4672,svc_backup,,,\n'
    '2026-03-14T06:43:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-14T12:50:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
))

EVIDENCIA = {}           # o que você vai analisar de fato (nome curto -> texto)
ORIGEM = {}              # de onde veio cada arquivo: "rede" ou "reserva"

for nome, reserva in RESERVA.items():
    texto = None
    if not FORCAR_OFFLINE:
        try:
            # timeout curto: se a rede não responder em 8 s, seguimos com o plano B
            with urllib.request.urlopen(URL_BASE + nome, timeout=8) as resp:
                texto = resp.read().decode("utf-8")
            ORIGEM[nome] = "rede"
        except Exception as erro:            # sem internet, proxy, 404... qualquer falha
            print(f"⚠️  {nome.split('/')[-1]}: rede indisponível ({type(erro).__name__}), usando a cópia embutida")
    if texto is None:
        texto = reserva
        ORIGEM[nome] = "reserva"
    EVIDENCIA[nome.split("/")[-1]] = texto   # guardamos pelo nome curto
    h = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    # Se o upstream mudou depois que este notebook foi gerado, os hashes divergem: avisamos.
    igual = "= cópia embutida" if texto == reserva else "≠ cópia embutida (o upstream mudou!)"
    print(f"{ORIGEM[nome]:8} {h[:16]}…  {nome.split('/')[-1]}  ({igual})")


## 1. O *spray* do `bastion01`

Cada comando do original, em Python, com o comando ao lado.


In [ ]:
import re
from collections import Counter

L = EVIDENCIA["bastion01-auth.log"].splitlines()

falhas = [l for l in L if "Failed password" in l]                       # grep 'Failed password'
print("grep -c 'Failed password':", len(falhas))

origens = Counter(re.search(r"from (\S+)", l).group(1) for l in falhas)  # awk ... print $(i+1) | uniq -c
print("origens:", dict(origens))

usuarios = Counter(re.search(r"Failed password for (?:invalid user )?(\S+)", l).group(1) for l in falhas)
print("usuários (uniq -c | sort -rn):")
for u, n in usuarios.most_common():
    print(f"   {n:>4} {u}")

print("primeira e última (sed -n '1p;$p' | cut -c1-15):", falhas[0][:15], "|", falhas[-1][:15])

print("\ngrep -E 'Accepted (password|publickey)':")
for l in L:
    if re.search(r"Accepted (password|publickey)", l):
        print("  ", l[:110])


### Passo 2: o que a linha de sucesso mostra


In [ ]:
sucesso = next(l for l in L if "Accepted password for svc_backup" in l)
anteriores_svc = [l for l in L if "Accepted" in l and "svc_backup" in l and l < sucesso]
print("Sucesso:", sucesso)
print("O svc_backup aparece em algum 'Accepted' anterior?", "sim" if anteriores_svc else "NÃO")
print("Método:", "senha" if "Accepted password" in sucesso else "chave")
print("\nMostra: login por SENHA de 203.0.113.45 numa conta sem histórico de login neste log.")
print("NÃO mostra: como a senha era conhecida.")

RESUMO_SPRAY = dict(origem=list(origens)[0], falhas=len(falhas), usuarios=len(usuarios),
                    por_usuario=set(usuarios.values()), primeira=falhas[0][:15], ultima=falhas[-1][:15],
                    sucesso=sucesso[:15])
print("\nResumo:", RESUMO_SPRAY)


## 2. Mapas de calor em texto

`.` = zero. Os números são contagens por dia × hora (UTC). Não precisa de cor: o olho acha o número sozinho.


In [ ]:
import csv, io
from datetime import datetime

def ler_seg(texto):
    return list(csv.DictReader(io.StringIO(texto)))

def mapa(eventos, eid, fuso_offset_h=0, titulo=""):
    """Grade dia x hora. fuso_offset_h = -3 para ver em Brasília (o Dia 56 faz isso direito, com zoneinfo)."""
    from datetime import timedelta
    c = Counter()
    for r in eventos:
        if r["EventID"] == str(eid) and r["TargetUserName"] != "SYSTEM":
            t = datetime.strptime(r["TimeCreatedUtc"], "%Y-%m-%dT%H:%M:%SZ") + timedelta(hours=fuso_offset_h)
            c[(t.strftime("%Y-%m-%d %a"), t.hour)] += 1
    dias = sorted({d for d, _ in c})
    print(f"\n{titulo} (EventID {eid})")
    print(" " * 15 + "".join(f"{h:>3}" for h in range(24)))
    for d in dias:
        print(f"{d:15}" + "".join(f"{c[(d, h)] or '.':>3}" for h in range(24)))
    return c

SEG = ler_seg(EVIDENCIA["ws-fin-07-security.csv"])
c_ok = mapa(SEG, 4624, titulo="WS-FIN-07 logons com sucesso, UTC")
c_falha = mapa(SEG, 4625, titulo="WS-FIN-07 logons com falha, UTC")

quente = max(c_falha, key=c_falha.get)
print(f"\nCélula quente das falhas: {quente[0]} hora {quente[1]:02d} -> {c_falha[quente]} falhas (o original espera 23)")


In [ ]:
# (OPCIONAL) O heatmap.py do original, com pandas e matplotlib (já vêm no Colab).
# Grava os PNGs; as anotações com setas você faz depois, num editor de imagem.
try:
    import pandas as pd
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    df = pd.read_csv(io.StringIO(EVIDENCIA["ws-fin-07-security.csv"]), parse_dates=["TimeCreatedUtc"])
    df = df[df["EventID"].isin([4624, 4625]) & (df["TargetUserName"] != "SYSTEM")]
    df["day"] = df["TimeCreatedUtc"].dt.strftime("%Y-%m-%d %a")
    df["hour"] = df["TimeCreatedUtc"].dt.hour
    for eid, title in [(4624, "Successful logons (4624)"), (4625, "Failed logons (4625)")]:
        grid = (df[df["EventID"] == eid]
                .pivot_table(index="day", columns="hour", values="EventID", aggfunc="count", fill_value=0)
                .reindex(columns=range(24), fill_value=0))
        fig, ax = plt.subplots(figsize=(12, 3.5))
        im = ax.imshow(grid.values, aspect="auto", cmap="Reds")
        ax.set_xticks(range(24)); ax.set_yticks(range(len(grid.index))); ax.set_yticklabels(grid.index)
        ax.set_xlabel("Hour of day (UTC)"); ax.set_title(f"WS-FIN-07 {title}, synthetic case LAB-P4")
        fig.colorbar(im, ax=ax); fig.tight_layout()
        fig.savefig(f"ws-fin-07-heatmap-{eid}.png", dpi=150)
        plt.close(fig)
    print("PNGs gravados: ws-fin-07-heatmap-4624.png e ws-fin-07-heatmap-4625.png")
except ImportError:
    print("(pandas/matplotlib ausentes: fique com o mapa em texto acima, ele tem a mesma informação)")


## 3. A janela 02h–04h (passo 4) e a questão em aberto

O `awk -F, '$1 >= "2026-03-14T02" && $1 < "2026-03-14T04"'` funciona porque datas ISO 8601 ordenam como texto. Em Python é a mesma comparação de strings.


In [ ]:
HOSTS = {"10.10.20.5": "bastion01", "10.10.30.17": "fs01", "10.10.40.57": "WS-FIN-07"}
janela = [r for r in SEG if "2026-03-14T02" <= r["TimeCreatedUtc"] < "2026-03-14T04"]
for r in janela:
    origem = r["IpAddress"]
    print(f"{r['TimeCreatedUtc']} {r['EventID']} {r['TargetUserName']:13} tipo {r['LogonType'] or '-':2} "
          f"de {origem or '-'} {('(' + HOSTS[origem] + ')') if origem in HOSTS else ''}")

f0210 = [r for r in janela if r["EventID"] == "4625"]
contas = sorted({r["TargetUserName"] for r in f0210})
questao = (f"QUESTÃO EM ABERTO: {len(f0210)} logons tipo 3 com falha contra o WS-FIN-07 vindos do bastion01 "
           f"(10.10.20.5), {f0210[0]['TimeCreatedUtc'][11:19]} a {f0210[-1]['TimeCreatedUtc'][11:19]} UTC, contas "
           f"{', '.join(contas)}. Estranho porque acontecem ANTES do spray contra o bastion01 (02:51:03) e partem "
           "de um host interno. Sem explicação por enquanto; sinalizado para o Dia 56.")
print("\n" + questao)


## 4. Classificador de origem

Para cada origem: quantas contas, quantas tentativas por conta. As regras são as do diagrama do conceito, mais a terceira forma que o caso BR precisa.


In [ ]:
def classificar(tentativas):
    """tentativas: lista de (origem, conta, sucesso_bool). Conta só as FALHAS para a forma; sucessos à parte."""
    falhas, sucessos = {}, Counter()
    for origem, conta, ok in tentativas:
        if ok:
            sucessos[origem] += 1
        else:
            falhas.setdefault(origem, Counter())[conta] += 1
    saida = {}
    for origem in sorted(set(falhas) | set(sucessos)):
        contas = falhas.get(origem, Counter())
        n_contas, total = len(contas), sum(contas.values())
        if total == 0:
            forma = "só sucessos (linha de base)"
        elif n_contas >= 5 and total / n_contas <= 20:
            forma = "PASSWORD SPRAY (muitas contas, poucas falhas cada)"
        elif n_contas <= 2 and total >= 20:
            forma = "FORÇA BRUTA (uma conta, muitas falhas)"
        elif n_contas == 1 and total <= 6:
            forma = "POUCAS FALHAS NUMA CONTA (credencial já conhecida? erro de digitação?)"
        else:
            forma = "indeterminado"
        saida[origem] = f"{n_contas} conta(s) com falha, {total} falhas, {sucessos[origem]} sucesso(s) -> {forma}"
    return saida

tent_bastion = []
for l in L:
    m = re.search(r"(Failed|Accepted) (?:password|publickey) for (?:invalid user )?(\S+) from (\S+)", l)
    if m:
        tent_bastion.append((m.group(3), m.group(2), m.group(1) == "Accepted"))
for origem, desc in classificar(tent_bastion).items():
    print(f"{origem:15} {desc}")


## 5. ➕ A semana da `cx.centro07`


In [ ]:
# ------------------------------------------------------------
# RECEBER A SEMANA DO SA-CX-07 E DO WEBMAIL
# Plano A: baixar do repositório original (GitHub raw), com urllib.
# Plano B: usar a cópia embutida logo abaixo (o arquivo inteiro,
#          copiado do upstream/), para o notebook rodar sem internet.
# ------------------------------------------------------------
import urllib.request      # baixa arquivos da web, sem instalar nada
import hashlib             # SHA-256: a impressão digital dos bytes

FORCAR_OFFLINE = False     # True = nem tenta a rede (útil em sala sem Wi-Fi)
URL_BASE = 'https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/'

RESERVA = {}             # caminho do arquivo -> conteúdo embutido (plano B)
RESERVA['dia-55/sa-cx-07-security-semana.csv'] = ''.join((
    'TimeCreatedUtc,EventID,TargetUserName,LogonType,IpAddress,WorkstationName\n'
    '2026-09-28T10:51:00Z,4624,SYSTEM,5,-,SA-CX-07\n'
    '2026-09-28T10:58:00Z,4624,cx.centro07,2,-,SA-CX-07\n'
    '2026-09-28T13:26:42Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-09-28T15:49:00Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-09-28T17:36:31Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-09-29T10:47:40Z,4625,cx.centro07,2,-,SA-CX-07\n'
    '2026-09-29T10:47:49Z,4624,cx.centro07,2,-,SA-CX-07\n'
    '2026-09-29T10:51:00Z,4624,SYSTEM,5,-,SA-CX-07\n'
    '2026-09-29T13:38:43Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-09-29T15:43:57Z,4625,cx.centro07,7,-,SA-CX-07\n'
    '2026-09-29T15:44:04Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-09-29T17:03:03Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-09-30T10:51:00Z,4624,SYSTEM,5,-,SA-CX-07\n'
    '2026-09-30T10:53:58Z,4624,cx.centro07,2,-,SA-CX-07\n'
    '2026-09-30T13:21:39Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-09-30T15:09:40Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-09-30T17:15:57Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-10-01T10:48:33Z,4624,cx.centro07,2,-,SA-CX-07\n'
    '2026-10-01T10:51:00Z,4624,SYSTEM,5,-,SA-CX-07\n'
    '2026-10-01T13:02:00Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-10-01T15:50:34Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-10-01T17:39:19Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-10-02T10:49:30Z,4624,cx.centro07,2,-,SA-CX-07\n'
    '2026-10-02T10:51:00Z,4624,SYSTEM,5,-,SA-CX-07\n'
    '2026-10-02T13:45:52Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-10-02T15:08:10Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-10-02T17:50:31Z,4624,cx.centro07,7,-,SA-CX-07\n'
    '2026-10-03T10:51:00Z,4624,SYSTEM,5,-,SA-CX-07\n'
    '2026-10-04T10:51:00Z,4624,SYSTEM,5,-,SA-CX-07\n'
    '2026-10-05T10:51:02Z,4624,SYSTEM,5,-,SA-CX-07\n'
    '2026-10-05T10:52:10Z,4624,cx.centro07,2,-,SA-CX-07\n'
    '2026-10-05T13:15:12Z,4625,cx.centro07,2,-,SA-CX-07\n'
    '2026-10-05T13:15:20Z,4624,cx.centro07,2,-,SA-CX-07\n'
))
RESERVA['dia-55/webmail-sa-semana.jsonl'] = ''.join((
    '{"ts": "2026-09-28T10:58:55Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "10.20.40.57", "result": "success"}\n'
    '{"ts": "2026-09-28T10:59:00Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "success"}\n'
    '{"ts": "2026-09-28T14:38:00Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "failure"}\n'
    '{"ts": "2026-09-28T14:38:18Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "success"}\n'
    '{"ts": "2026-09-29T10:48:31Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "10.20.40.57", "result": "success"}\n'
    '{"ts": "2026-09-29T10:54:00Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "success"}\n'
    '{"ts": "2026-09-30T10:55:14Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "10.20.40.57", "result": "success"}\n'
    '{"ts": "2026-09-30T10:57:00Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "success"}\n'
    '{"ts": "2026-09-30T17:52:00Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "failure"}\n'
    '{"ts": "2026-09-30T17:52:18Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "success"}\n'
    '{"ts": "2026-10-01T10:49:14Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "10.20.40.57", "result": "success"}\n'
    '{"ts": "2026-10-01T10:56:00Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "success"}\n'
    '{"ts": "2026-10-02T10:51:00Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "10.20.40.57", "result": "success"}\n'
    '{"ts": "2026-10-02T10:54:00Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "success"}\n'
    '{"ts": "2026-10-02T14:40:00Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "failure"}\n'
    '{"ts": "2026-10-02T14:40:18Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "success"}\n'
    '{"ts": "2026-10-05T10:53:02Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "10.20.40.57", "result": "success"}\n'
    '{"ts": "2026-10-05T10:54:41Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "success"}\n'
    '{"ts": "2026-10-05T12:31:02Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "failure"}\n'
    '{"ts": "2026-10-05T12:31:15Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "failure"}\n'
    '{"ts": "2026-10-05T12:31:40Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "failure"}\n'
    '{"ts": "2026-10-05T12:32:05Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "success"}\n'
    '{"ts": "2026-10-05T12:33:50Z", "event": "mailbox.rule_create", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "success"}\n'
    '{"ts": "2026-10-05T13:05:01Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "failure"}\n'
    '{"ts": "2026-10-05T13:05:19Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "success"}\n'
))

SEMANA = {}           # o que você vai analisar de fato (nome curto -> texto)
ORIGEM = {}              # de onde veio cada arquivo: "rede" ou "reserva"

for nome, reserva in RESERVA.items():
    texto = None
    if not FORCAR_OFFLINE:
        try:
            # timeout curto: se a rede não responder em 8 s, seguimos com o plano B
            with urllib.request.urlopen(URL_BASE + nome, timeout=8) as resp:
                texto = resp.read().decode("utf-8")
            ORIGEM[nome] = "rede"
        except Exception as erro:            # sem internet, proxy, 404... qualquer falha
            print(f"⚠️  {nome.split('/')[-1]}: rede indisponível ({type(erro).__name__}), usando a cópia embutida")
    if texto is None:
        texto = reserva
        ORIGEM[nome] = "reserva"
    SEMANA[nome.split("/")[-1]] = texto   # guardamos pelo nome curto
    h = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    # Se o upstream mudou depois que este notebook foi gerado, os hashes divergem: avisamos.
    igual = "= cópia embutida" if texto == reserva else "≠ cópia embutida (o upstream mudou!)"
    print(f"{ORIGEM[nome]:8} {h[:16]}…  {nome.split('/')[-1]}  ({igual})")


In [ ]:
SEG_SA = ler_seg(SEMANA["sa-cx-07-security-semana.csv"])
mapa(SEG_SA, 4624, 0, "SA-CX-07 logons com sucesso, UTC")
mapa(SEG_SA, 4624, -3, "SA-CX-07 logons com sucesso, BRASÍLIA (UTC-3)")
tipos = Counter((r["LogonType"], r["TimeCreatedUtc"][11:13]) for r in SEG_SA if r["EventID"] == "4624" and r["TargetUserName"] != "SYSTEM")
print("\n(tipo, hora UTC):", dict(sorted(tipos.items())))
print("Linha de base: tipo 2 por volta das 10h50 UTC (07h50 em Brasília) e desbloqueios tipo 7 ao longo do dia.")
print("Nenhum logon anormal no notebook: a intrusão do caso não passou pela autenticação do SA-CX-07.")


In [ ]:
import json
WEB = [json.loads(l) for l in SEMANA["webmail-sa-semana.jsonl"].splitlines() if l.strip()]

print("Origens da cx.centro07 (grep | uniq -c):")
print(dict(Counter(r["src_ip"] for r in WEB if r["user"] == "cx.centro07")))

tent_web = [(r["src_ip"], r["user"], r["result"] == "success") for r in WEB if r["event"] == "auth.login"]
print("\nClassificação por origem:")
for origem, desc in classificar(tent_web).items():
    print(f"  {origem:15} {desc}")

print("\nO que aconteceu a partir de 203.0.113.91:")
for r in WEB:
    if r["src_ip"] == "203.0.113.91":
        print(f"  {r['ts']}  {r['event']:20} {r['user']}  {r.get('result', '')}")

captura = datetime(2026, 10, 5, 11, 23, 12)      # último POST/CONNECT ao site falso no proxy (Dia 49)
primeiro = datetime.strptime(next(r["ts"] for r in WEB if r["src_ip"] == "203.0.113.91"), "%Y-%m-%dT%H:%M:%SZ")
print(f"\nIntervalo entre a última conexão ao site falso e a 1ª tentativa externa: {(primeiro - captura).seconds // 60} min")
print("O log NÃO mostra a senha usada nem como foi obtida. A regra de caixa postal: criada; conteúdo não registrado.")


> 🔎 Repare que o classificador também acusa 10.20.40.52 (`ger.centro01`) com "poucas falhas numa conta". É a gerente errando a senha e acertando 18 s depois, em vários dias, de dentro da agência. A regra é a mesma; o **contexto** (origem interna conhecida, linha de base, repetição) é o que separa. Classificador é ponto de partida, não veredito.

## 6. 🎮 Quiz dos tipos de *logon*


In [ ]:
MODO_JOGO = True
QUIZ = [("RDP", "10"), ("rede (SMB, ferramentas remotas)", "3"), ("desbloqueio de tela", "7"),
        ("interativo (teclado)", "2"), ("serviço", "5"), ("credenciais em cache", "11")]
acertos = 0
for i, (nome, certo) in enumerate(QUIZ, 1):
    if MODO_JOGO:
        r = input(f"[{i}] LogonType de {nome}: ").strip()
        acertos += r == certo
        print("   ✅" if r == certo else f"   ❌ é {certo}")
    else:
        print(f"[{i}] {nome}: {certo}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}  (checkpoint: RDP, rede e desbloqueio de memória)")


In [ ]:
from pathlib import Path
Path("notes").mkdir(exist_ok=True)
Path("notes/day55-auth.md").write_text(
    "# Dia 55\n\n## Spray no bastion01\n" + "\n".join(f"- {k}: {v}" for k, v in RESUMO_SPRAY.items()) +
    "\n\n## Questão em aberto\n" + questao + "\n", encoding="utf-8")
print(Path("notes/day55-auth.md").read_text(encoding="utf-8"))
try:
    from google.colab import files
    files.download("notes/day55-auth.md")
except ImportError:
    print("(Fora do Colab: notes/day55-auth.md)")


## ✅ Antes de fechar o Dia 55

- [ ] resumo do *spray*: origem, 112 falhas, 8 usuários × 14, 02:51:03 / 03:13:36 / 03:14:07
- [ ] mapa 4625 com a célula quente (23 falhas, hora 02, sábado 14) anotada como vinda do `bastion01`
- [ ] questão em aberto das 02:10, sem explicação
- [ ] ➕ linha de base da `cx.centro07` em UTC e Brasília; webmail com tabela de hipóteses

**Amanhã (Dia 56):** quatro relógios, um só UTC. O firewall da Serra Azul grava em Brasília sem dizer, e o proxy não diz o ano.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>
